# Restauración y algoritmos de inpainting

En este cuaderno vas a simular daños sobre una imagen y después vas a comparar dos algoritmos de `inpainting`. La idea es entender qué tipo de problema intentan resolver y qué tan plausible resulta la reconstrucción.


## Objetivo

Construir una imagen dañada de manera controlada y comparar cómo la restauran dos métodos de OpenCV: Telea y Navier-Stokes.

## Resultados de aprendizaje

Al finalizar este cuaderno, vas a poder:

- crear una máscara de daño sintética;
- aplicar `cv2.inpaint()` con dos variantes;
- comparar resultados globales y locales;
- discutir límites de plausibilidad en restauración automática.

## Relación con la secuencia

Este cuaderno se apoya en el trabajo previo con máscaras y limpieza. Acá la máscara ya no sirve para segmentar un objeto, sino para indicar qué región debe ser reconstruida.


## Módulos que vamos a usar

- `cv2`: para crear la máscara y aplicar los algoritmos de inpainting.
- `numpy`: para construir regiones dañadas.
- `matplotlib.pyplot`: para comparar resultados.
- `pathlib.Path`: para abrir la imagen de trabajo.


In [ ]:
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

ruta_imagen = Path("Imagenes") / "building.jpg"
imagen_bgr = cv2.imread(str(ruta_imagen), cv2.IMREAD_COLOR)
if imagen_bgr is None:
    raise FileNotFoundError(f"No se pudo leer la imagen: {ruta_imagen}")

imagen_rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)
alto, ancho = imagen_rgb.shape[:2]


## 1. Imagen original y construcción del daño

Vamos a crear un daño sintético sobre la imagen original. Esto es útil porque conocemos la escena previa y podemos comparar mejor qué tan razonable resulta la reconstrucción.


In [ ]:
mascara_dano = np.zeros((alto, ancho), dtype=np.uint8)
cv2.line(mascara_dano, (90, 120), (470, 120), 255, 10)
cv2.line(mascara_dano, (200, 60), (230, 320), 255, 8)
cv2.rectangle(mascara_dano, (330, 210), (470, 280), 255, -1)

imagen_danada_bgr = imagen_bgr.copy()
imagen_danada_bgr[mascara_dano == 255] = (255, 255, 255)
imagen_danada_rgb = cv2.cvtColor(imagen_danada_bgr, cv2.COLOR_BGR2RGB)


In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
ejes[0].imshow(imagen_rgb)
ejes[0].set_title("Original", fontweight="bold", loc="left")
ejes[0].axis("off")
ejes[1].imshow(mascara_dano, cmap="gray")
ejes[1].set_title("Máscara de daño", fontweight="bold", loc="left")
ejes[1].axis("off")
ejes[2].imshow(imagen_danada_rgb)
ejes[2].set_title("Imagen dañada", fontweight="bold", loc="left")
ejes[2].axis("off")
plt.show()


## 2. Aplicar dos métodos de inpainting

OpenCV ofrece dos variantes conocidas. Las vamos a usar con la misma máscara para ver cómo cambia la reconstrucción.


In [ ]:
imagen_telea_bgr = cv2.inpaint(imagen_danada_bgr, mascara_dano, 3, cv2.INPAINT_TELEA)
imagen_ns_bgr = cv2.inpaint(imagen_danada_bgr, mascara_dano, 3, cv2.INPAINT_NS)

imagen_telea_rgb = cv2.cvtColor(imagen_telea_bgr, cv2.COLOR_BGR2RGB)
imagen_ns_rgb = cv2.cvtColor(imagen_ns_bgr, cv2.COLOR_BGR2RGB)


In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
ejes[0].imshow(imagen_danada_rgb)
ejes[0].set_title("Imagen dañada", fontweight="bold", loc="left")
ejes[0].axis("off")
ejes[1].imshow(imagen_telea_rgb)
ejes[1].set_title("Inpainting Telea", fontweight="bold", loc="left")
ejes[1].axis("off")
ejes[2].imshow(imagen_ns_rgb)
ejes[2].set_title("Inpainting Navier-Stokes", fontweight="bold", loc="left")
ejes[2].axis("off")
plt.show()


## 3. Mirar una región local

En restauración, la comparación global no siempre alcanza. Una zona recortada ayuda a ver si las líneas continúan de forma plausible o si la reconstrucción deja marcas artificiales.


In [ ]:
fila_inicial, fila_final = 80, 310
columna_inicial, columna_final = 70, 320

recorte_danado = imagen_danada_rgb[fila_inicial:fila_final, columna_inicial:columna_final]
recorte_telea = imagen_telea_rgb[fila_inicial:fila_final, columna_inicial:columna_final]
recorte_ns = imagen_ns_rgb[fila_inicial:fila_final, columna_inicial:columna_final]

fig, ejes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
ejes[0].imshow(recorte_danado)
ejes[0].set_title("Recorte dañado", fontweight="bold", loc="left")
ejes[0].axis("off")
ejes[1].imshow(recorte_telea)
ejes[1].set_title("Recorte Telea", fontweight="bold", loc="left")
ejes[1].axis("off")
ejes[2].imshow(recorte_ns)
ejes[2].set_title("Recorte Navier-Stokes", fontweight="bold", loc="left")
ejes[2].axis("off")
plt.show()


Acá la pregunta importante no es si el algoritmo “adivinó” exactamente el original, sino si logró una reconstrucción verosímil para la zona dañada. En restauración automática, esa diferencia es clave.


## Actividad breve

Modificá la máscara de daño para que afecte otra zona de la imagen y repetí el experimento. Después describí:

1. qué estructura era más difícil de reconstruir;
2. cuál de los dos métodos te resultó más convincente;
3. por qué restaurar no es lo mismo que recuperar información exacta.


## Cierre

Los algoritmos de inpainting pueden producir resultados visualmente razonables, pero no recuperan mágicamente la información perdida. Lo que hacen es proponer una reconstrucción plausible a partir del contexto cercano. Por eso conviene evaluarlos siempre con criterio visual y técnico.
